Memory:
    - Allows  a chain to remember  past interactions and use  them in future responses

Why memory needed:
    - Multi turn conversations
    - chatbots
    - agents
    - context continuity
    - personalization

Memory  != Model memory:
    - llms are stateless
    - memory is handled by langchain
    - memory = prompt augmentation

Types of Memory:
    1] ConversationBufferMemory
    2] ConversationBufferWindowMemory
    3] ConversationSummaryMemory
    4] ConversationSummaryBufferMemory
    5] ConversationEntityMemory
    6] VectorStoreMemoory
    7] CombinedMemory
    8] ReadOnlySharedMemory


| Memory Type   | Stores             | Best For            |
| ------------- | ------------------ | ------------------- |
| Buffer        | Full chat          | Short conversations |
| Window        | Last K turns       | Token control       |
| Summary       | Compressed history | Long chats          |
| SummaryBuffer | Hybrid             | Production          |
| Entity        | Facts/entities     | Personalization     |
| VectorStore   | Semantic recall    | Long-term memory    |
| Combined      | Multiple memories  | Complex flows       |
| ReadOnly      | Shared read-only   | Multi-chain         |


Memory:

1] memory.buffer
    full history conversation of it 
2] memory.chat_memory.messages
    list of messages in objects 
3] memory.load_memory_variables({})
    output : {'history': "conversation...."}
    shows exact output
4] memory.save_context()
    syntax:
     memory.save_context({"question": "question text.."}, {"answer": "answer text..."})
    manual injection not using LLLChain()
5] memory.clear()
    clear whole conversation until u have been done or saved
6] memory.moving_summary_buffer                                                            
    Auto generates the summary of older chats
7] memory.chat_memory.clear()
    only recent conversation or messages
8] memory.memory_key
    confirm prompt variable like we have used here {history}
9] memory.input_key
    confirm user input {question} or {chat} 
10]memory.output_key
    needed for multi-output chains 
11] memory.return_messages
    True -> message object result
    False -> text string

In [ ]:
Wokring of Every Memory:
    step 1: load model
    step 2: use memory
    step 3: prompt template
    step 4: chain
    step 5: result

In [ ]:
1] ConversationBufferMemory
- stores entire conversation history and sends it back to llm on everynew request
- No summarization, no truncation, rawconversation text

wokring:
    1] user sends input
    2] memory appends it
    3] memory injects full history into prompt
    4] llm responds using full context

Arch:

User → Input
       ↓
ConversationBufferMemory
       ↓
Prompt (history + new input)
       ↓
LLM → Response

When to use it:
    Learning and debugging
    short conversation
    simplechatbots


In [ ]:
ConversationBufferMemory: parameters :
    memory_key='history', #name of the variable where history is stored
    input_key = None, # tells which input will be the one form the user eg.question
    output_key = None,  # specific answer which is from the chain to store
    return_message = False,  # defaule is false : plain text :true , message object(human message, ai message)
    human_prefix = "Human", # prefix for user eg.human, user, question
    ai_prefix = "Ai" #prefix for ai generatd answer like ai, answer, chat


LLMChain : parameters:
    llm=llm, (required)
    prompt = prompt, (required)
    memory = memory, (required)
    output_key= 'text'
    verbose = False,
    callbacks = None

In [22]:
#ConversationBufferMemory
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from langchain.llms import HuggingFacePipeline
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
pipe = pipeline("text-generation", model=model,tokenizer=tokenizer,max_new_tokens=30,temperature=0.9,do_sample=True, pad_token_id= tokenizer.eos_token_id)
llm = HuggingFacePipeline(pipeline=pipe)


#memory
memory = ConversationBufferMemory()

#prompt
template  = """
You are a simple assistant.
Answer briefly and clearly.

Conversation:
{history}
------------------
Question:
{question}

Answer:

"""

prompt = PromptTemplate(input_variables=['history','question'], template=template)

#chain
chain = LLMChain(llm=llm, prompt=prompt,memory=memory)

print(chain.invoke({"question": "Hello"}))
print(chain.invoke({"question": "my name is sumit"}))
print(chain.invoke({"question": "what is my name ??"}))

Device set to use cpu


{'question': 'Hello', 'history': '', 'text': "\nYou are a simple assistant.\nAnswer briefly and clearly.\n\nConversation:\n\n------------------\nQuestion:\nHello\n\nAnswer:\n\nHi,\nThis isn't a good question.\nAnswer very few times. It's a question that you ask a long time ago. If you"}
{'question': 'my name is sumit', 'history': "Human: Hello\nAI: \nYou are a simple assistant.\nAnswer briefly and clearly.\n\nConversation:\n\n------------------\nQuestion:\nHello\n\nAnswer:\n\nHi,\nThis isn't a good question.\nAnswer very few times. It's a question that you ask a long time ago. If you", 'text': "\nYou are a simple assistant.\nAnswer briefly and clearly.\n\nConversation:\nHuman: Hello\nAI: \nYou are a simple assistant.\nAnswer briefly and clearly.\n\nConversation:\n\n------------------\nQuestion:\nHello\n\nAnswer:\n\nHi,\nThis isn't a good question.\nAnswer very few times. It's a question that you ask a long time ago. If you\n------------------\nQuestion:\nmy name is sumit\n\nAnswer:\n\n

In [ ]:
from langchain.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.memory import ConversationBufferMemory
from langchain.chains import LLMChain
from transformers import pipeline,AutoTokenizer, AutoModelForCausalLM

#load model
model_name = "tiiuae/Falcon-H1-Tiny-Multilingual-100M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer,max_new_tokens=50,use_fast=True,torch_dtype='int8',pad_token_id=tokenizer.eos_token_id)
llm = HuggingFacePipeline(pipeline=pipe)

#memory
memory = ConversationBufferMemory(memory_key='history', input_key='question', return_message=False, human_prefix='User', ai_prefix='Answer')

#prompt
template = """
History:
{history}

User: {question}
Answer:
"""

prompt  = PromptTemplate(input_variables=['history', 'question'],template=template)

#chain
chain = LLMChain(llm=llm, prompt=prompt, memory=memory, verbose=True)

print(chain.invoke({"question": "What is ai"}))
print(chain.invoke({"question": "what was my previous question"}))

Loading weights:   0%|          | 0/386 [00:00<?, ?it/s]

Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




> Entering new LLMChain chain...
Prompt after formatting:

History:


User: What is ai
Answer: 



Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



> Finished chain.
{'question': 'What is ai', 'history': '', 'text': '\nHistory:\n\n\nUser: What is ai\nAnswer: \nAI is a type of artificial intelligence that acts as a translator between humans and machines. It enables machines to learn from and make decisions based on patterns in the data it has been trained on. AI can perform a wide range of tasks, including recognizing'}


> Entering new LLMChain chain...
Prompt after formatting:

History:
User: What is ai
Answer: 
History:


User: What is ai
Answer: 
AI is a type of artificial intelligence that acts as a translator between humans and machines. It enables machines to learn from and make decisions based on patterns in the data it has been trained on. AI can perform a wide range of tasks, including recognizing

User: what was my previous question
Answer: 


> Finished chain.
{'question': 'what was my previous question', 'history': 'User: What is ai\nAnswer: \nHistory:\n\n\nUser: What is ai\nAnswer: \nAI is a type of artificial intell

In [ ]:
2] ConversationBufferWindowMemory
It stores only last k conversation turns instead of the full chat history
- remember only recent content forget older stuff
IF  k =2
User: Q1
AI: A1
User: Q2
AI: A2
User: Q3
AI: A3   ← memory now contains only Q2, A2, Q3, A3

This is useful when :
    - token limit matter,
    - model starts hallucinating with long history
    - you want  focused short conversations

In [2]:
#ConversationBufferWindowMemory
from langchain.llms import HuggingFacePipeline
from langchain.memory import ConversationBufferWindowMemory
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, FalconH1ForCausalLM

#load model
model_name = 'tiiuae/Falcon-H1-Tiny-Multilingual-100M-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = FalconH1ForCausalLM.from_pretrained(model_name)
pipe = pipeline("text-generation",model=model,
                tokenizer=tokenizer, max_new_tokens=50,
                dtype='int8',
                use_fast=True, )
llm = HuggingFacePipeline(pipeline=pipe)

#memory
memory = ConversationBufferWindowMemory(k=2, memory_key='history', input_key='question',return_messages=False, human_prefix="User", ai_prefix="Answer")


#prompt

template = """
History:
{history}

User: {question}
Answer
"""

prompt = PromptTemplate(input_variables=["history","question"], template=template)

#chain
chain = LLMChain(llm=llm, memory=memory, prompt=prompt, verbose=True)

print(chain.invoke({"question": "What is Ai "})['text'])
print(chain.invoke({"question": "Explain in simply"})['text'])
print(chain.invoke({"question": "What i was my first question"})['text'])

Loading weights:   0%|          | 0/386 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




> Entering new LLMChain chain...
Prompt after formatting:

History:


User: What is Ai 
Answer



Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



> Finished chain.

History:


User: What is Ai 
Answer
AI:
"Ai" is a term from the field of AI technology. It refers to a set of algorithms or instructions used to process and generate text. In the context of AI, "Ai" can refer to various aspects such as natural


> Entering new LLMChain chain...
Prompt after formatting:

History:
User: What is Ai 
Answer: 
History:


User: What is Ai 
Answer
AI:
"Ai" is a term from the field of AI technology. It refers to a set of algorithms or instructions used to process and generate text. In the context of AI, "Ai" can refer to various aspects such as natural

User: Explain in simply
Answer



Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



> Finished chain.

History:
User: What is Ai 
Answer: 
History:


User: What is Ai 
Answer
AI:
"Ai" is a term from the field of AI technology. It refers to a set of algorithms or instructions used to process and generate text. In the context of AI, "Ai" can refer to various aspects such as natural

User: Explain in simply
Answer
AI (AI):
AI (Artificial Intelligence) refers to the simulation of human intelligence in machines that are programmed to think and learn like humans. It involves the development of algorithms and machine learning models that can perform tasks that typically require


> Entering new LLMChain chain...
Prompt after formatting:

History:
User: What is Ai 
Answer: 
History:


User: What is Ai 
Answer
AI:
"Ai" is a term from the field of AI technology. It refers to a set of algorithms or instructions used to process and generate text. In the context of AI, "Ai" can refer to various aspects such as natural
User: Explain in simply
Answer: 
History:
User: What is Ai 
An

In [ ]:
3] ConversaionSummaryMemory:
    Instead of storing  raw  conversation, it stores a running  sumaryb of coonversation
    
think of it as:
    full chat hisotry nope
    last k mesage nope
    compressed meaning of conversation yes
    
eg.

Conversation:
    User: What is AI?
    AI: AI is the simulation of human intelligence...
    User: Give examples
    AI: Chatbots, recommendation systems...

Stored memory:
        User  asked  about ai its defination and example like chatbots and recommendations
        
When yoou should use it:
    - long conversation
    - small models 
    - avoiding token explosion
    - maintaing  context  without confusion
    
Parameters:
ConversationSummaryMemory(
    llm,
    memory_key="history",
    input_key=None,
    output_key=None,
    return_messages=False,
    buffer="",                 # explain about somthing eg. its about calling agent service so the model will wokr mostly to that way only
    human_prefix="Human",
    ai_prefix="AI",
    summary_prompt=None
)

In [2]:
# ConversationSummaryMemory
from langchain.llms import HuggingFacePipeline
from langchain.memory import ConversationSummaryMemory
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from transformers import pipeline , AutoTokenizer, AutoModelForCausalLM

#load model
model_name = 'tiiuae/Falcon-H1-Tiny-Multilingual-100M-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
pipe = pipeline("text-generation", model=model, max_new_tokens=30,tokenizer=tokenizer, use_fast=True, dtype='int8')
llm = HuggingFacePipeline(pipeline=pipe)

#memory 
memory = ConversationSummaryMemory(llm=llm, memory_key='history', input_key='question', return_messages=False)

#prompt
template = """
Conversation summary so far:
{history}
------------------------------

User:  {question}
Answer: 
"""

prompt = PromptTemplate(input_variables=['history','question'], template=template)

#chain 
chain = LLMChain(llm=llm, memory=memory, prompt=prompt, verbose=True)

print(chain.invoke({"question":"what is artifical intelliegnce"})['text'])
print(chain.invoke({"question":"give simple examples "})['text'])
print(chain.invoke({"question":"what were  we discussing"})['text'])

The fast path is not available because one of `(selective_state_update, causal_conv1d_fn, causal_conv1d_update)` is None. Falling back to the naive implementation. To install follow https://github.com/state-spaces/mamba/#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/386 [00:00<?, ?it/s]

Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




> Entering new LLMChain chain...
Prompt after formatting:

Conversation summary so far:

------------------------------

User:  what is artifical intelliegnce
Answer: 



Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



> Finished chain.

Conversation summary so far:

------------------------------

User:  what is artifical intelliegnce
Answer: 
Artifical intelliegnices are a concept in computer graphics and artificial intelligence, referring to AI that is designed to perform complex visual tasks in


> Entering new LLMChain chain...
Prompt after formatting:

Conversation summary so far:
Progressively summarize the lines of conversation provided, adding onto the previous summary returning a new summary.

EXAMPLE
Current summary:
The human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good.

New lines of conversation:
Human: Why do you think artificial intelligence is a force for good?
AI: Because artificial intelligence will help humans reach their full potential.

New summary:
The human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good because it will help humans reach their full po

Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



> Finished chain.

Conversation summary so far:
Progressively summarize the lines of conversation provided, adding onto the previous summary returning a new summary.

EXAMPLE
Current summary:
The human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good.

New lines of conversation:
Human: Why do you think artificial intelligence is a force for good?
AI: Because artificial intelligence will help humans reach their full potential.

New summary:
The human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good because it will help humans reach their full potential.
END OF EXAMPLE

Current summary:


New lines of conversation:
Human: what is artifical intelliegnce
AI: 
Conversation summary so far:

------------------------------

User:  what is artifical intelliegnce
Answer: 
Artifical intelliegnices are a concept in computer graphics and artificial intelligence, referring to AI

Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



> Finished chain.

Conversation summary so far:
Progressively summarize the lines of conversation provided, adding onto the previous summary returning a new summary.

EXAMPLE
Current summary:
The human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good.

New lines of conversation:
Human: Why do you think artificial intelligence is a force for good?
AI: Because artificial intelligence will help humans reach their full potential.

New summary:
The human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good because it will help humans reach their full potential.
END OF EXAMPLE

Current summary:
Progressively summarize the lines of conversation provided, adding onto the previous summary returning a new summary.

EXAMPLE
Current summary:
The human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good.

New lines of conver

In [3]:
print(memory.buffer)

Progressively summarize the lines of conversation provided, adding onto the previous summary returning a new summary.

EXAMPLE
Current summary:
The human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good.

New lines of conversation:
Human: Why do you think artificial intelligence is a force for good?
AI: Because artificial intelligence will help humans reach their full potential.

New summary:
The human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good because it will help humans reach their full potential.
END OF EXAMPLE

Current summary:
Progressively summarize the lines of conversation provided, adding onto the previous summary returning a new summary.

EXAMPLE
Current summary:
The human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good.

New lines of conversation:
Human: Why do you think artificial intell

In [ ]:
4] ConversationSummaryBufferMemory:
- Important memory type 
- keeps recent message  verbatim
- older message are summarized  automaically
- Prevents context  overflow 
- Much better for small models 

Why you should use it:
    - Repeated blank outputs : Summary compress context
    - Small models forgets : Summaryis short and focused
    - Long history noise : Old chat summarized
    - Memory grows infinitely : Token limited buffer

Wokring:
    Recent Chat (buffer)
↓
When buffer exceeds limit
↓
Old messages → summarized using LLM
↓
Summary stored + new messages added

Parameters:
    1] llm 
    2] max_token_limit: int value with numbers to get the ouptput limted 
    3] input_key
    4] output_key
    5] return_messages 
    6] human_prefix
    7] ai_prefix

In [ ]:
from transformers import pipeline, AutoTokenizer,  AutoModelForCausalLM
from langchain.llms import HuggingFacePipeline
from langchain.memory import  ConversationSummaryBufferMemory
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

model_name = "tiiuae/Falcon-H1-Tiny-Multilingual-100M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=30, dtype="int8", use_fast=True)
llm = HuggingFacePipeline(pipeline=pipe)

#memory
memory = ConversationSummaryBufferMemory(llm=llm, max_token_limit=30, memory_key='history', input_key="question",return_messages=True)

#prompt
template = """
Conversation so far:
{history}

User: {question}
Answer: 
"""

prompt = PromptTemplate(input_variables = ['history', 'question'], template=template)

#chain 
chain = LLMChain(llm=llm, memory=memory, prompt=prompt, verbose=True)

#Result 
questions = [
    'What  is Artifical Intelligence',
    'Give simple example',
    'Explain it in one line',
    'What were we dicsussing',
]

for q in questions:
    result = chain.invoke({"question":q})
    print(f"\nQ: {q}")
    print(f"A: {result['text']}")

The fast path is not available because one of `(selective_state_update, causal_conv1d_fn, causal_conv1d_update)` is None. Falling back to the naive implementation. To install follow https://github.com/state-spaces/mamba/#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/386 [00:00<?, ?it/s]

Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




> Entering new LLMChain chain...
Prompt after formatting:

Conversation so far:
[]

User: What  is Artifical Intelligence
Answer: 



Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



> Finished chain.

Q: Give simple example
A: 
Conversation so far:
[SystemMessage(content='Progressively summarize the lines of conversation provided, adding onto the previous summary returning a new summary.\n\nEXAMPLE\nCurrent summary:\nThe human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good.\n\nNew lines of conversation:\nHuman: Why do you think artificial intelligence is a force for good?\nAI: Because artificial intelligence will help humans reach their full potential.\n\nNew summary:\nThe human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good because it will help humans reach their full potential.\nEND OF EXAMPLE\n\nCurrent summary:\n\n\nNew lines of conversation:\nHuman: What  is Artifical Intelligence\nAI: \nConversation so far:\n[]\n\nUser: What  is Artifical Intelligence\nAnswer: \nArtifical Intelligence is a sub-process for AI that is integrated into t

Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



> Finished chain.

Q: Explain it in one line
A: 
Conversation so far:
[SystemMessage(content="Progressively summarize the lines of conversation provided, adding onto the previous summary returning a new summary.\n\nEXAMPLE\nCurrent summary:\nThe human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good.\n\nNew lines of conversation:\nHuman: Why do you think artificial intelligence is a force for good?\nAI: Because artificial intelligence will help humans reach their full potential.\n\nNew summary:\nThe human asks what the AI thinks of artificial intelligence. The AI thinks artificial intelligence is a force for good because it will help humans reach their full potential.\nEND OF EXAMPLE\n\nCurrent summary:\nProgressively summarize the lines of conversation provided, adding onto the previous summary returning a new summary.\n\nEXAMPLE\nCurrent summary:\nThe human asks what the AI thinks of artificial intelligence. The AI thinks 

Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Token indices sequence length is longer than the specified maximum sequence length for this model (1975 > 1024). Running this sequence through the model will result in indexing errors
Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
5] ConversationEntityMemory:
- detects entities in conversation
- stores facts about those entities
- injects those facts back into future prompts

What does it stores :
names, roles , places, preference ,relationships
It doesnt store whole conversation

When to use it :
    Personalize chabots
    CRM, HR bots 
    Assistants that must rememeber facts

Working :
    uses llm to :
        extract entites, updates entity facts
    Stores entity summaries
    injects enity info into future prompts


In [2]:
from langchain.llms import HuggingFacePipeline
from langchain.memory import ConversationEntityMemory
from langchain.prompts import PromptTemplate
from transformers import pipeline,  AutoModelForCausalLM, AutoTokenizer
from langchain.chains import LLMChain

model_name = "tiiuae/Falcon-H1-Tiny-Multilingual-100M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
pipe = pipeline("text-generation", model=model,tokenizer=tokenizer, use_fast=True, dtype="int8",max_new_tokens=30)
llm = HuggingFacePipeline(pipeline=pipe)

#memory
memory = ConversationEntityMemory(llm=llm, memory_key="entity_memory", input_key="input")

#inject data in memory
memory.save_context(
    {'input': 'My name is Sumit'},
    {'output': 'Nice to meet you sumit'},
)
memory.save_context(
    {'input': 'I am a Gen ai developer and I use python'},
    {'output': 'Thats great'},
)

print(memory.load_memory_variables({'input':'What do you know about me '}) )

Loading weights:   0%|          | 0/386 [00:00<?, ?it/s]

Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'history': 'Human: My name is Sumit\nAI: Nice to meet you sumit\nHuman: I am a Gen ai developer and I use python\nAI: Thats great', 'entities': {'You are an AI assistant reading the transcript of a conversation between an AI and a human. Extract all of the proper nouns from the last line of conversation. As a guideline': '', 'a proper noun is generally capitalized. You should definitely extract all names and places.\n\nThe conversation history is provided just in case of a coreference (e.g. "What do you know about him" where "him" is defined in a previous line) -- ignore items mentioned there that are not in the last line.\n\nReturn the output as a single comma-separated list': '', 'or NONE if there is nothing of note to return (e.g. the user is just issuing a greeting or having a simple conversation).\n\nEXAMPLE\nConversation history:\nPerson #1: how\'s it going today?\nAI: "It\'s going great! How about you?"\nPerson #1: good! busy working on Langchain. lots to do.\nAI: "That sounds 

In [ ]:
6] VectorStoreMemory:
Stores past conversations as emdedding  and retrives only the most relevant ones based on semantic similarity

Usecases:
Long conversations
You dont want token explosion
You want  semantic recall
Chat history mattersbut not all of it

Working:
User message
   ↓
Convert to embedding
   ↓
Store in Vector DB
   ↓
New query → similarity search
   ↓
Relevant past messages injected into prompt



Paramters:
1] vectorstores (required)
    stores embdeddding past messages 
    suppport FAISS, Chroma etc
2]memory_key
3] input_key
4] return_docs:
    False: return string
    True: returns Document []
5] k 
6] retriever_kwargs :
    "search_type": "similarity"
    "search_kwargs": {k:3}

In [7]:
from langchain.llms import HuggingFacePipeline
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.memory import VectorStoreRetrieverMemory
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.schema import Document
from langchain.schema.runnable import RunnableSequence
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer

#step 1: load llm model
hf_pipeline = pipeline(
    "text-generation",
    model="distilgpt2",
    max_new_tokens=40
)
llm = HuggingFacePipeline(pipeline=hf_pipeline)

#step 2: load embedding model
emdedding = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")

#step 3: vectorstore
doc = [
    Document(page_content="Inital Memory", metadata={"source":"init"})       # memory unit / page_conent: actual text, metadata: extra info(source, timestamp, etx)
]
vectorstore = FAISS.from_documents(doc,emdedding)       #stroes data in numercial values

# step 4: memory
retriever = vectorstore.as_retriever(search_kwargs={"k":3}, search_type="similarity")     # search the relevant history of the user if input and source data similar memory
memory = VectorStoreRetrieverMemory(retriever=retriever, memory_key="chat_history")

#step 6: prompt
prompt = PromptTemplate(
    input_variables=["chat_history", "input"],
    template="""
You are a helpful AI assistant.

Relevant past conversation:
{chat_history}

User: {input}
Assistant:
"""
)

#step 7: chain 
chain = LLMChain(llm=llm , memory=memory, prompt=prompt, verbose=True)


#step 8: Run
response1 = chain.invoke({"input": "My name is Sumit"})
print(response1["text"])

response2 = chain.invoke({"input": "I am a GenAI developer using Python"})
print(response2["text"])

response3 = chain.invoke({"input": "What do you remember about me?"})
print(response3["text"])


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




> Entering new LLMChain chain...
Prompt after formatting:

You are a helpful AI assistant.

Relevant past conversation:
Inital Memory

User: My name is Sumit
Assistant:



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



> Finished chain.

You are a helpful AI assistant.

Relevant past conversation:
Inital Memory

User: My name is Sumit
Assistant:
You are a helpful AI assistant.
Relevant past conversation:
Inital Memory
User: my name is Sumit.
Relevant past conversation:
Inital Memory
User: My


> Entering new LLMChain chain...
Prompt after formatting:

You are a helpful AI assistant.

Relevant past conversation:
input: My name is Sumit
text: 
You are a helpful AI assistant.

Relevant past conversation:
Inital Memory

User: My name is Sumit
Assistant:
You are a helpful AI assistant.
Relevant past conversation:
Inital Memory
User: my name is Sumit.
Relevant past conversation:
Inital Memory
User: My
Inital Memory

User: I am a GenAI developer using Python
Assistant:



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



> Finished chain.

You are a helpful AI assistant.

Relevant past conversation:
input: My name is Sumit
text: 
You are a helpful AI assistant.

Relevant past conversation:
Inital Memory

User: My name is Sumit
Assistant:
You are a helpful AI assistant.
Relevant past conversation:
Inital Memory
User: my name is Sumit.
Relevant past conversation:
Inital Memory
User: My
Inital Memory

User: I am a GenAI developer using Python
Assistant:
You are a helpful AI assistant.
Relevant past conversation:
Inital Memory
User: My name is Sumit
Assistant:
You are a helpful AI assistant.
Relevant past


> Entering new LLMChain chain...
Prompt after formatting:

You are a helpful AI assistant.

Relevant past conversation:
Inital Memory
input: My name is Sumit
text: 
You are a helpful AI assistant.

Relevant past conversation:
Inital Memory

User: My name is Sumit
Assistant:
You are a helpful AI assistant.
Relevant past conversation:
Inital Memory
User: my name is Sumit.
Relevant past conversation:
Init

In [ ]:
7] ReadOnlyMemory:
multiple chains read the same memory, but prevvents  from  modifying it
- memory can be shared but cannot written only read

Architecture:
Writable memory -> ReadOnlyMemory -> Multiple chians 


In [34]:
from langchain.llms import HuggingFacePipeline
from langchain.memory import ConversationBufferMemory, ReadOnlySharedMemory
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from transformers import pipeline

pipe = pipeline("text-generation", model='tiiuae/Falcon-H1-Tiny-Multilingual-100M-Instruct', max_new_tokens=50, use_fast=True,temperature=0.7)
llm = HuggingFacePipeline(pipeline=pipe)


base_memory = ConversationBufferMemory(memory='history', return_messsages=True)

base_memory.save_context(
    {"input": "My name is WarHammer"},
    {"output": "Nice to meet you WarHammer"}
)
base_memory.save_context(
    {"input": "I am an great Warrior"},
    {"output": "That's great way to express yourself"}
    )

read_only_memory = ReadOnlySharedMemory(memory= base_memory)


tempalte = """
Conversation:
{history}


User: {input}
Ai: 
"""

prompt = PromptTemplate(input_variables= ['history','input'], template=tempalte)

chain = LLMChain(llm=llm, memory=read_only_memory, prompt=prompt)

#result 
response = chain.run("What is my name ")
print(response)

Loading weights:   0%|          | 0/386 [00:00<?, ?it/s]

Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Conversation:
Human: My name is WarHammer
AI: Nice to meet you WarHammer
Human: I am an great Warrior
AI: That's great way to express yourself


User: What is my name 
Ai: 
Human: nice to meet you WarHammer
AI: Nice to meet you WarHammer
Human: Hi, I am WarHammer, your name is WarHammer
AI: Hi, I am War
